# Vesuvius Surface Detection — Inference & Visualization

Visual comparison of predictions from:
1. **Custom 3D U-Net**
2. **nnU-Net** (Default, LPlans, XLPlans, MPlans)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
os.chdir('..')

import numpy as np
import torch
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load a sample volume + ground truth

In [ ]:
# Pick a validation sample (scroll_id=26002)
import csv

val_ids = []
with open('data/train.csv') as f:
    for row in csv.DictReader(f):
        if row['scroll_id'] == '26002':
            img_path = Path(f'data/train_images/{row["id"]}.tif')
            if img_path.exists():
                val_ids.append(row['id'])

print(f'Found {len(val_ids)} validation volumes')
SAMPLE_ID = val_ids[0]
print(f'Using sample: {SAMPLE_ID}')

image = tifffile.imread(f'data/train_images/{SAMPLE_ID}.tif')
label = tifffile.imread(f'data/train_labels/{SAMPLE_ID}.tif')
print(f'Image: {image.shape}, Label: {label.shape}, Label classes: {np.unique(label)}')

## 2. Custom 3D U-Net Inference

In [ ]:
from src.models.unet3d import UNet3D
from src.utils.utils import load_config, get_gaussian_3d

cfg = load_config('configs/unet3d_config.yaml')

# Load model
model_unet = UNet3D(
    in_channels=1, num_classes=3, base_channels=32,
    channel_multipliers=[1,2,4,8,10], num_conv_per_stage=2,
    deep_supervision=True
).to(device)

ckpt = torch.load('checkpoints/checkpoint_best.pth', map_location=device, weights_only=False)
model_unet.load_state_dict(ckpt['model_state_dict'])
model_unet.eval()
print(f'Loaded 3D U-Net from epoch {ckpt["epoch"]}, best_surface_dice={ckpt["best_surface_dice"]:.4f}')

In [ ]:
def sliding_window_inference(model, volume_np, patch_size=128, overlap=0.5, num_classes=3):
    """Run inference on a full 320^3 volume using overlapping patches."""
    model.eval()
    volume = torch.from_numpy(volume_np.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(device)
    
    step = int(patch_size * (1 - overlap))
    D, H, W = volume.shape[2:]
    gaussian = get_gaussian_3d(patch_size).to(device)
    
    output_sum = torch.zeros(num_classes, D, H, W, device=device)
    weight_sum = torch.zeros(1, D, H, W, device=device)
    
    d_starts = list(range(0, max(D - patch_size, 0) + 1, step))
    h_starts = list(range(0, max(H - patch_size, 0) + 1, step))
    w_starts = list(range(0, max(W - patch_size, 0) + 1, step))
    if d_starts[-1] + patch_size < D: d_starts.append(D - patch_size)
    if h_starts[-1] + patch_size < H: h_starts.append(H - patch_size)
    if w_starts[-1] + patch_size < W: w_starts.append(W - patch_size)
    
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for d in d_starts:
            for h in h_starts:
                for w in w_starts:
                    patch = volume[:, :, d:d+patch_size, h:h+patch_size, w:w+patch_size]
                    logits = model(patch)['logits']
                    probs = torch.softmax(logits.float(), dim=1)[0]
                    output_sum[:, d:d+patch_size, h:h+patch_size, w:w+patch_size] += probs * gaussian
                    weight_sum[:, d:d+patch_size, h:h+patch_size, w:w+patch_size] += gaussian
    
    probs_full = (output_sum / weight_sum.clamp(min=1e-8))
    pred = probs_full.argmax(dim=0).cpu().numpy()
    surface_prob = probs_full[1].cpu().numpy()  # class 1 = surface
    return pred, surface_prob

print('Running 3D U-Net inference...')
pred_unet, surf_prob_unet = sliding_window_inference(model_unet, image)
print(f'Prediction classes: {np.unique(pred_unet)}')

# Free GPU memory
del model_unet
torch.cuda.empty_cache()

## 3. nnU-Net Inference

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

os.environ['nnUNet_raw'] = 'nnUNet_data/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = 'nnUNet_data/nnUNet_preprocessed'
os.environ['nnUNet_results'] = 'nnUNet_data/nnUNet_results'

def run_nnunet_inference(model_dir, volume_np, folds='all'):
    """Run nnU-Net inference on a single volume."""
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=True,
        device=device,
        verbose=False,
    )
    predictor.initialize_from_trained_model_folder(
        model_dir,
        use_folds=(folds,) if isinstance(folds, str) else folds,
        checkpoint_name='checkpoint_best.pth',
    )
    
    # nnU-Net expects (C, D, H, W) float32 input
    img = volume_np.astype(np.float32)[np.newaxis]  # (1, D, H, W)
    props = {'spacing': [1.0, 1.0, 1.0]}
    
    pred, probs = predictor.predict_single_npy_array(img, props, None, None, True)
    surface_prob = probs[1] if probs.shape[0] > 1 else probs[0]
    return pred, surface_prob

print('Available nnU-Net models:')
results_dir = Path('nnUNet_data/nnUNet_results/Dataset200_VesuviusSurface')
for d in sorted(results_dir.iterdir()):
    if d.is_dir():
        ckpts = list(d.rglob('checkpoint_best.pth'))
        status = 'has checkpoint_best' if ckpts else 'no checkpoint'
        print(f'  {d.name} -- {status}')

In [ ]:
# Run nnU-Net Default (fully trained, 1000 epochs)
print('Running nnU-Net Default inference...')
pred_default, surf_prob_default = run_nnunet_inference(
    str(results_dir / 'nnUNetTrainer__nnUNetPlans__3d_fullres'), image, folds='all'
)
print(f'Default prediction classes: {np.unique(pred_default)}')
torch.cuda.empty_cache()

In [ ]:
# Run nnU-Net MPlans (best surface dice among running models)
print('Running nnU-Net MPlans inference...')
pred_mplans, surf_prob_mplans = run_nnunet_inference(
    str(results_dir / 'nnUNetTrainer_4000epochs__nnUNetResEncUNetMPlans__3d_fullres'), image, folds='all'
)
print(f'MPlans prediction classes: {np.unique(pred_mplans)}')
torch.cuda.empty_cache()

## 4. Compute Metrics

In [ ]:
from src.training.metrics import SegmentationMetrics

def compute_metrics(pred, gt):
    m = SegmentationMetrics(num_classes=3, class_names=['air', 'surface', 'papyrus'])
    m.update(torch.from_numpy(pred), torch.from_numpy(gt))
    return m.compute()

models = {
    '3D U-Net (custom)': pred_unet,
    'nnU-Net Default (1000ep)': pred_default,
    'nnU-Net MPlans (best)': pred_mplans,
}

print(f'{"Model":<30} {"Mean Dice":>10} {"Surface Dice":>13} {"Air Dice":>10} {"Papyrus Dice":>13} {"Voxel Acc":>10}')
print('-' * 90)
for name, pred in models.items():
    r = compute_metrics(pred, label)
    print(f'{name:<30} {r["mean_dice"]:>10.4f} {r["surface_dice"]:>13.4f} '
          f'{r["dice_per_class"]["air"]:>10.4f} {r["dice_per_class"]["papyrus"]:>13.4f} '
          f'{r["voxel_accuracy"]:>10.4f}')

## 5. Visualization — Axial Slices

In [ ]:
# Color map: 0=black (air), 1=red (surface), 2=blue (papyrus)
seg_cmap = ListedColormap(['black', 'red', 'dodgerblue'])

def plot_comparison(image, label, predictions, slice_fracs=[0.25, 0.5, 0.75], axis=0):
    """Plot axial slices comparing GT vs all model predictions."""
    n_slices = len(slice_fracs)
    n_cols = 2 + len(predictions)  # image, GT, then each model
    fig, axes = plt.subplots(n_slices, n_cols, figsize=(4 * n_cols, 4 * n_slices))
    
    col_titles = ['CT Image', 'Ground Truth'] + list(predictions.keys())
    
    for row, frac in enumerate(slice_fracs):
        idx = int(frac * image.shape[axis])
        img_slice = np.take(image, idx, axis=axis)
        lbl_slice = np.take(label, idx, axis=axis)
        
        axes[row, 0].imshow(img_slice, cmap='gray')
        axes[row, 0].set_ylabel(f'Slice {idx}', fontsize=12)
        
        axes[row, 1].imshow(img_slice, cmap='gray', alpha=0.5)
        axes[row, 1].imshow(lbl_slice, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=2)
        
        for col, (name, pred) in enumerate(predictions.items(), start=2):
            pred_slice = np.take(pred, idx, axis=axis)
            axes[row, col].imshow(img_slice, cmap='gray', alpha=0.5)
            axes[row, col].imshow(pred_slice, cmap=seg_cmap, alpha=0.5, vmin=0, vmax=2)
        
        for col in range(n_cols):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
    
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, fontsize=13, fontweight='bold')
    
    plt.suptitle(f'Sample {SAMPLE_ID} — Axial Slices', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'notebooks/comparison_axial_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_comparison(image, label, models)

## 6. Surface Probability Heatmaps

In [ ]:
def plot_surface_probability(image, label, prob_maps, slice_frac=0.5, axis=0):
    """Plot surface class probability heatmaps."""
    idx = int(slice_frac * image.shape[axis])
    n_cols = 1 + len(prob_maps)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    
    # Ground truth surface mask
    lbl_slice = np.take(label, idx, axis=axis)
    img_slice = np.take(image, idx, axis=axis)
    axes[0].imshow(img_slice, cmap='gray', alpha=0.5)
    axes[0].imshow(lbl_slice == 1, cmap='Reds', alpha=0.6)
    axes[0].set_title('GT Surface', fontsize=13, fontweight='bold')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    
    for i, (name, prob) in enumerate(prob_maps.items(), start=1):
        prob_slice = np.take(prob, idx, axis=axis)
        im = axes[i].imshow(prob_slice, cmap='hot', vmin=0, vmax=1)
        axes[i].set_title(name, fontsize=13, fontweight='bold')
        axes[i].set_xticks([]); axes[i].set_yticks([])
        plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
    
    plt.suptitle(f'Surface Probability — Slice {idx}', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'notebooks/surface_prob_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
    plt.show()

prob_maps = {
    '3D U-Net': surf_prob_unet,
    'nnU-Net Default': surf_prob_default,
    'nnU-Net MPlans': surf_prob_mplans,
}
plot_surface_probability(image, label, prob_maps, slice_frac=0.5)

## 7. Per-Slice Dice Score

In [ ]:
def per_slice_dice(pred, gt, target_class=1, axis=0):
    """Compute Dice score per slice along an axis."""
    n_slices = gt.shape[axis]
    dices = []
    for i in range(n_slices):
        p = np.take(pred, i, axis=axis) == target_class
        g = np.take(gt, i, axis=axis) == target_class
        intersection = (p & g).sum()
        union = p.sum() + g.sum()
        dice = (2 * intersection) / (union + 1e-8) if union > 0 else 1.0
        dices.append(dice)
    return np.array(dices)

fig, ax = plt.subplots(1, 1, figsize=(12, 5))
for name, pred in models.items():
    dices = per_slice_dice(pred, label, target_class=1)
    ax.plot(dices, label=f'{name} (mean={dices.mean():.3f})', linewidth=1.5)

ax.set_xlabel('Slice Index (Axial)', fontsize=12)
ax.set_ylabel('Surface Dice', fontsize=12)
ax.set_title(f'Per-Slice Surface Dice — Sample {SAMPLE_ID}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'notebooks/per_slice_dice_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 3D Surface Rendering (Optional)

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from skimage import measure

def plot_3d_surface(volume, title='Surface', downsample=4, threshold=0.5):
    """3D render of surface class using marching cubes."""
    mask = (volume == 1).astype(np.float32)
    # Downsample for speed
    mask_ds = mask[::downsample, ::downsample, ::downsample]
    
    if mask_ds.sum() < 10:
        print(f'{title}: too few surface voxels to render')
        return
    
    verts, faces, _, _ = measure.marching_cubes(mask_ds, level=threshold)
    
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    mesh = Poly3DCollection(verts[faces], alpha=0.3, edgecolor='none', facecolor='red')
    ax.add_collection3d(mesh)
    ax.set_xlim(0, mask_ds.shape[0])
    ax.set_ylim(0, mask_ds.shape[1])
    ax.set_zlim(0, mask_ds.shape[2])
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('Ground Truth surface:')
plot_3d_surface(label, 'Ground Truth Surface')

print('nnU-Net Default prediction:')
plot_3d_surface(pred_default, 'nnU-Net Default Surface')

## 9. Summary

In [ ]:
print('=' * 70)
print('  TRAINING SUMMARY')
print('=' * 70)
print()
print('Models trained:')
print('  1. Custom 3D U-Net  — residual encoder, deep supervision, DDP x4 GPUs')
print('  2. nnU-Net Default  — PlainConvUNet, 1000 epochs (fully trained)')
print('  3. nnU-Net MPlans   — ResEncUNet M, 4000 epochs (partial)')
print('  4. nnU-Net LPlans   — ResEncUNet L, 4000 epochs (partial)')
print('  5. nnU-Net XLPlans  — ResEncUNet XL, 250 epochs (partial)')
print()
print('Dataset: 786 volumes (320x320x320 uint8), 3 classes')
print('  Class 0: Air (~25%)')
print('  Class 1: Surface boundary (~4%) — the key target')
print('  Class 2: Papyrus interior (~71%)')
print()
print('Metrics on sample volume:')
for name, pred in models.items():
    r = compute_metrics(pred, label)
    print(f'  {name:<30}  Surface Dice: {r["surface_dice"]:.4f}  Mean Dice: {r["mean_dice"]:.4f}')